# 03 - Real Data: DermaMNIST
Goal: load real dermatoscope images, understand the samples and labels, see the class imbalance,
and build a Dataset + DataLoader that feeds the model in batches.

In [1]:
from medmnist import DermaMNIST

# download=True fetches it once; 28x28 images to start (CPU-friendly)
train_data = DermaMNIST(split="train", download=True, size=28)

print(train_data)

100%|██████████| 19.7M/19.7M [00:09<00:00, 2.16MB/s]


Dataset DermaMNIST of size 28 (dermamnist)
    Number of datapoints: 7007
    Root location: C:\Users\study\.medmnist
    Split: train
    Task: multi-class
    Number of channels: 3
    Meaning of labels: {'0': 'actinic keratoses and intraepithelial carcinoma', '1': 'basal cell carcinoma', '2': 'benign keratosis-like lesions', '3': 'dermatofibroma', '4': 'melanoma', '5': 'melanocytic nevi', '6': 'vascular lesions'}
    Number of samples: {'train': 7007, 'val': 1003, 'test': 2005}
    Description: The DermaMNIST is based on the HAM10000, a large collection of multi-source dermatoscopic images of common pigmented skin lesions. The dataset consists of 10,015 dermatoscopic images categorized as 7 different diseases, formulized as a multi-class classification task. We split the images into training, validation and test set with a ratio of 7:1:2. The source images of 3×600×450 are resized into 3×28×28.
    License: CC BY-NC 4.0


## What DermaMNIST is
- 7007 train / 1003 val / 2005 test images, split 7:1:2 (val = held-out "real exam")
- RGB images, shape (3, 28, 28), so 3 x 28 x 28 = 2352 input values (not 784 - TinyNet needs updating)
- 7 classes = skin lesion diagnoses. Class 5 (nevi) common, class 4 (melanoma) rare -> class imbalance
- Source: HAM10000 dermatoscopy dataset

In [2]:
import numpy as np

labels = train_data.labels.flatten()   # the label for every training image
classes, counts = np.unique(labels, return_counts=True)

for c, n in zip(classes, counts):
    pct = 100 * n / len(labels)
    print(f"class {c}: {n:5d}  ({pct:4.1f}%)")

class 0:   228  ( 3.3%)
class 1:   359  ( 5.1%)
class 2:   769  (11.0%)
class 3:    80  ( 1.1%)
class 4:   779  (11.1%)
class 5:  4693  (67.0%)
class 6:    99  ( 1.4%)


## Class imbalance (the core challenge)
Counts: class 5 = 67%, class 3 = 1.1%. A 59x gap between most and least common.
- A model that ALWAYS predicts class 5 gets 67% accuracy while learning nothing -> accuracy is misleading here.
- The rare classes include class 4 (melanoma), the most dangerous, so ignoring rare classes is exactly the wrong failure.
- Plan: judge by per-class recall / F1 (not accuracy), and fight imbalance with weighted loss, then measure what helps.